# UpstageDocumentParseLoader (구 UpstageLayoutAnalysisLoader, 2026년 최신 권장 사용법)

Upstage 의 문서 분석 API 를 LangChain 로더로 사용하는 방법입니다.

**주요 특징:**
- PDF, 이미지, Office 문서 등 다양한 형식의 문서에서 레이아웃 분석 수행
- 문서의 구조적 요소(제목, 단락, 표, 그림, 차트 등)를 자동으로 인식 및 추출
- OCR 지원, 결과를 HTML / Markdown / Text 로 출력

> **⚠️ 2026년 9월 기준 변경 사항**
>
> - Upstage 의 **Layout Analysis API 는 Document Parse API 로 대체**되었고, `langchain-upstage` 에서도 `UpstageLayoutAnalysisLoader` 대신 **`UpstageDocumentParseLoader`** 를 사용합니다.
> - `langchain-upstage` 는 LangChain 공식 통합 목록의 PDF 로더로 등재된 **전용 통합 패키지**이므로 `langchain-community` sunset 과 무관합니다.
> - 책의 `langchain_teddynote.logging.langsmith(...)` 는 LangSmith **환경변수 설정**으로 대체합니다.
>
> **파라미터 대응표**
>
> | `UpstageLayoutAnalysisLoader` (구) | `UpstageDocumentParseLoader` (현재) |
> |---|---|
> | `output_type="html"` / `"text"` | `output_format="html"` / `"text"` / `"markdown"` |
> | `use_ocr=True` | `ocr="force"` (기본 `"auto"`: PDF 는 내장 텍스트 사용) |
> | `split="none"` / `"element"` / `"page"` | 동일 |
> | `exclude=["header", "footer"]` | **없음** → `split="element"` 로 받은 뒤 `metadata["category"]` 로 직접 필터링 |
> | — | `coordinates`, `chart_recognition`, `base64_encoding=["table", ...]`, `model` 추가 |

**설치**

```bash
pip install -U langchain-upstage python-dotenv
```

**API Key 설정**

- Upstage API KEY 발급 링크: https://console.upstage.ai/
- `.env` 파일에 `UPSTAGE_API_KEY` 키를 설정합니다.

**참고**
- [Upstage Document Parse 문서](https://console.upstage.ai/docs/capabilities/document-parse)
- [UpstageDocumentParseLoader API 레퍼런스](https://reference.langchain.com/python/langchain-upstage/document_parse/UpstageDocumentParseLoader)

## 환경 설정

In [ ]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

### LangSmith 추적 설정 (구 `langchain_teddynote.logging.langsmith`)

`.env` 에 아래 값을 넣어 두면 별도 코드 없이 추적이 활성화됩니다.

```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=CH07-DocumentLoader
```

> 참고: 문서 로더 자체는 Runnable 이 아니어서 추적 대상이 아닙니다. 이후 챕터(체인/에이전트)에서 추적 결과를 확인하게 됩니다.

In [ ]:
import os

# 코드에서 프로젝트 이름만 바꾸고 싶을 때
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "CH07-DocumentLoader"

## UpstageDocumentParseLoader

**주요 파라미터**
- `file_path`: 분석할 문서 경로 (여러 개 리스트 가능)
- `output_format`: 출력 형식 [(기본값) `"html"`, `"markdown"`, `"text"`]
- `split`: 문서 분할 방식 [`"none"`(기본), `"element"`, `"page"`]
- `ocr`: `"auto"`(기본) / `"force"`(OCR 강제)
- `coordinates`: 요소 좌표 포함 여부 (기본 `True`)
- `chart_recognition`: 차트를 표 데이터로 인식 (기본 `True`)
- `base64_encoding`: 지정한 카테고리(예: `["table", "figure"]`) 요소의 이미지를 base64 로 함께 반환

In [ ]:
from langchain_upstage import UpstageDocumentParseLoader

# 파일 경로
file_path = "./data/SPRI_AI_Brief_2023년12월호_F.pdf"

# 문서 로더 설정
loader = UpstageDocumentParseLoader(
    file_path,
    output_format="html",
    split="element",
    ocr="force",  # 구 use_ocr=True
)

# 문서 로드
all_elements = loader.load()

# 구 exclude=["header", "footer"] 를 직접 구현
EXCLUDE = {"header", "footer"}
docs = [d for d in all_elements if d.metadata.get("category") not in EXCLUDE]

print(len(all_elements), "→", len(docs))

# 결과 출력
for doc in docs[:3]:
    print(doc)

In [ ]:
docs[11]

In [ ]:
from collections import Counter

# 어떤 요소 카테고리가 있는지 확인
Counter(d.metadata.get("category") for d in all_elements)

## Markdown 출력 + 페이지 단위 분할

RAG 에서는 Markdown 이 토큰 효율이 좋고, 이후 `MarkdownHeaderTextSplitter` 로 섹션 분할하기도 쉽습니다.

In [ ]:
loader = UpstageDocumentParseLoader(file_path, output_format="markdown", split="page")
page_docs = loader.load()

print(len(page_docs))
print(page_docs[1].page_content[:800])

## 표만 뽑아서 사용하기

`split="element"` 결과에서 `category == "table"` 인 요소만 모으면 표 전용 인덱스를 만들 수 있습니다. `base64_encoding=["table"]` 을 주면 표 이미지를 함께 받아 멀티모달 모델에 넘길 수도 있습니다.

In [ ]:
table_docs = [d for d in all_elements if d.metadata.get("category") == "table"]
print(len(table_docs))
if table_docs:
    print(table_docs[0].page_content[:1000])